# Generate the hexagonal grid

Builds the regular hexagonal grid used for aggregation, from the national boundary, at a
chosen **point-to-point spacing** (8.5 km in the published dataset). Writes a GeoPackage
with a unique `hex_id`, the full hexagon area, and the area falling within the country.

This is the provenance step for the grid consumed by `aggregate_to_hexgrid_mean.ipynb`.

*Requires:* `geopandas numpy shapely pyproj`.

In [ ]:
# ============================ CONFIG ============================
BOUNDARY   = "cz_boundary.gpkg"     # national boundary (any polygon vector: gpkg/shp/geojson)
BOUNDARY_LAYER = None               # layer name if the boundary file has several; else None
SPACING_KM = 8.5                    # hexagon point-to-point distance (vertex to opposite vertex)
TARGET_CRS = 5514                   # S-JTSK / Krovak East-North
OUT_GPKG   = "hexgrid.gpkg"
OUT_LAYER  = "hexgrid_8p5km"
MIN_COVERAGE = 0.0                  # keep hexes whose in-country area fraction exceeds this (0 = any overlap)
# ===============================================================

In [ ]:
import numpy as np, geopandas as gpd, sqlite3, os
from shapely.geometry import Polygon
from shapely.prepared import prep
from pyproj import CRS

boundary = gpd.read_file(BOUNDARY, layer=BOUNDARY_LAYER) if BOUNDARY_LAYER else gpd.read_file(BOUNDARY)
boundary = boundary.to_crs(TARGET_CRS)
land = boundary.union_all() if hasattr(boundary,"union_all") else boundary.unary_union
minx, miny, maxx, maxy = land.bounds
print("boundary CRS:", boundary.crs.to_epsg(), "| bbox km:",
      round((maxx-minx)/1000), "x", round((maxy-miny)/1000))

## Build the hexagons
Pointy-top hexagons on a staggered grid. For point-to-point spacing *s*, the circumradius is
*R = s/2*; column spacing = √3·R, row spacing = 1.5·R, with alternate rows offset by √3·R/2.

In [ ]:
s = SPACING_KM*1000.0
R = s/2.0                          # circumradius (centre -> vertex)
dx = np.sqrt(3)*R                  # horizontal centre spacing
dy = 1.5*R                         # vertical centre spacing
ang = np.deg2rad([30,90,150,210,270,330])
ux, uy = R*np.cos(ang), R*np.sin(ang)   # pointy-top vertex offsets

def hexagon(cx, cy):
    return Polygon(zip(cx+ux, cy+uy))

# staggered centres covering the bbox (+1 ring margin)
rows=[]; j=0; y=miny-dy
while y <= maxy+dy:
    xoff = (dx/2.0) if (j%2) else 0.0
    x = minx-dx + xoff
    while x <= maxx+dx:
        rows.append((x,y)); x += dx
    y += dy; j += 1
print(f"candidate hexagon centres: {len(rows):,}")

PL = prep(land)
recs=[]
for cx,cy in rows:
    poly = hexagon(cx,cy)
    if not poly.intersects(land):    # quick reject
        continue
    inter = poly.intersection(land)
    if inter.is_empty or inter.area == 0:
        continue
    frac = inter.area/poly.area
    if frac <= MIN_COVERAGE:
        continue
    recs.append({"geometry":poly,
                 "area_km2":round(poly.area/1e6,4),
                 "area_cz_km2":round(inter.area/1e6,4)})
grid = gpd.GeoDataFrame(recs, crs=TARGET_CRS)
grid = grid.sort_values(["geometry"], key=lambda s:[ -g.centroid.y*1e7 + g.centroid.x for g in s]).reset_index(drop=True)
grid.insert(0,"hex_id",[f"H{i+1:05d}" for i in range(len(grid))])
print(f"hexagons kept: {len(grid):,}  | mean area {grid.area_km2.mean():.1f} km^2  "
      f"(theoretical {(3*np.sqrt(3)/2)*(R/1000)**2:.1f})")

## Write GeoPackage (CRS stamped as ESRI WKT1 for ArcGIS)

In [ ]:
if os.path.exists(OUT_GPKG): os.remove(OUT_GPKG)
grid.to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG")
con=sqlite3.connect(OUT_GPKG)
con.execute("UPDATE gpkg_spatial_ref_sys SET definition=? WHERE srs_id=? OR organization_coordsys_id=?",
            (CRS.from_epsg(TARGET_CRS).to_wkt(version="WKT1_ESRI"), TARGET_CRS, TARGET_CRS))
con.commit(); con.close()
print(f"wrote {OUT_GPKG} (layer '{OUT_LAYER}', {len(grid)} hexagons, EPSG:{TARGET_CRS} as ESRI WKT1)")

### Notes
- `SPACING_KM` is point-to-point (vertex to opposite vertex); hexagon area ≈ (3√3/2)·(s/2)². At 8.5 km this is ≈ 47 km².
- `area_cz_km2` is the in-country area, used to compute densities near the border correctly.
- `MIN_COVERAGE` > 0 drops slivers that barely clip the border; 0 keeps every hexagon touching the country.